# 03 — Gold: business aggregates

Reads `silver.orders` and the current rows of `silver.dim_customer`. No quality rules — gold
assumes silver's contract holds, and asserts it rather than re-deriving it. One customer with
two open dimension rows would double their revenue with no error and no null.

In [0]:
from pyspark.sql import functions as F

CATALOG       = "retail"
SILVER_ORDERS = f"{CATALOG}.silver.orders"
SILVER_DIM    = f"{CATALOG}.silver.dim_customer"
GOLD_DAILY    = f"{CATALOG}.gold.daily_revenue"
GOLD_CUSTOMER = f"{CATALOG}.gold.customer_revenue"

orders = spark.table(SILVER_ORDERS)
dim    = spark.table(SILVER_DIM)

print("silver.orders      :", orders.count())
print("dim_customer rows  :", dim.count())
print("dim_customer open  :", dim.filter("is_current").count())

# The one input assumption gold cannot survive being wrong about.
dupes = (dim.filter("is_current").groupBy("customer_id")
            .count().filter("count > 1").count())
assert dupes == 0, f"{dupes} customer_id(s) have two open rows — every join would double them"

orders.printSchema()

## The revenue basis

Revenue excludes `CANCELLED` — a business definition, and the first thing to check when a gold
total disagrees with someone's spreadsheet. Defined once as `billable` and reused by both gold
tables, so they cannot drift apart.

In [0]:
EXCLUDED_STATUSES = ["CANCELLED"]

# Gold doesn't clean. It refuses to run if silver didn't.
contract = orders.agg(
    F.sum(F.col("amount").isNull().cast("int")).alias("null_amount"),
    F.sum((F.col("amount") < 0).cast("int")).alias("negative_amount"),
    F.sum(F.col("order_ts").isNull().cast("int")).alias("null_order_ts"),
    F.sum(F.col("status").isNull().cast("int")).alias("null_status"),
).collect()[0].asDict()

print("contract check     :", contract)
assert all(v == 0 for v in contract.values()), f"silver contract broken: {contract}"

billable = orders.filter(~F.col("status").isin(EXCLUDED_STATUSES))

print("silver.orders      :", orders.count())
print("excluded           :", orders.count() - billable.count())
print("billable           :", billable.count())

total_revenue = billable.agg(F.sum("amount")).collect()[0][0]
print("billable revenue   :", total_revenue)

display(orders.groupBy("status").count().orderBy(F.col("count").desc()))

## `gold.daily_revenue`

Full recompute, overwritten each run — this table rebuilds from `silver.orders` in seconds, so
overwrite beats a merge. At volume that flips to `replaceWhere` on the affected dates.
Grain is `to_date(order_ts)`: when the order happened, not when the row was last touched.

In [0]:
daily = (billable
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("amount").alias("revenue"),
        F.countDistinct("customer_id").alias("distinct_customers"),
    ))

(daily.write.format("delta")
      .mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(GOLD_DAILY))

d = spark.table(GOLD_DAILY)

print("days                 :", d.count())
print("orders across days   :", d.agg(F.sum("order_count")).collect()[0][0], "(expect 421)")
print("revenue across days  :", d.agg(F.sum("revenue")).collect()[0][0])
print("max distinct/day     :", d.agg(F.max("distinct_customers")).collect()[0][0], "(<= 65)")

display(d.orderBy("order_date"))

## `gold.customer_revenue`

One row per **current** customer, not per customer who billed something. `WHERE revenue > 0`
recovers the narrower table; nothing recovers the wider one. Joining `is_current` only is a
**Type 1 read of a Type 2 dimension** — right for "best customers today", wrong for "what did
Mumbai bill in July".

In [0]:
dim_current = (dim.filter("is_current")
                  .select("customer_id", "customer_name", "city", "segment", "country"))

joined = dim_current.join(billable, on="customer_id", how="left")

n_billable = billable.count()
n_facts    = joined.filter(F.col("order_id").isNotNull()).count()
print("billable orders      :", n_billable)
print("fact rows after join :", n_facts)
assert n_facts == n_billable, (
    f"join changed the fact count {n_billable} -> {n_facts}: "
    "orphan customer_id, or a duplicate dimension key")

ZERO = F.lit(0).cast("decimal(22,2)")

customer_rev = (joined
    .groupBy("customer_id", "customer_name", "city", "segment", "country")
    .agg(
        F.count("order_id").alias("order_count"),   # not count("*"): the left join gives a no-order customer one null-filled row
        F.coalesce(F.sum("amount"), ZERO).alias("revenue"),
        F.min(F.to_date("order_ts")).alias("first_order_date"),
        F.max(F.to_date("order_ts")).alias("last_order_date"),
    ))

(customer_rev.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(GOLD_CUSTOMER))

c = spark.table(GOLD_CUSTOMER)

print("rows                    :", c.count(), "(expect 65)")
print("with revenue > 0        :", c.filter("revenue > 0").count(), "(expect 60)")
print("orders across customers :", c.agg(F.sum("order_count")).collect()[0][0], "(expect 421)")
print("revenue across customers:", c.agg(F.sum("revenue")).collect()[0][0])

display(c.filter("revenue = 0"))
display(c.orderBy(F.col("revenue").desc()).limit(10))

## Reconciliation

Two tables, two code paths — one groups by date, the other joins a dimension and groups by
customer. Agreement with silver and with each other means the join neither lost nor multiplied
a row. `decimal(12,2)` makes this exact equality; on `double` it would decay into "close enough".

In [0]:
s_tot = (spark.table(SILVER_ORDERS)
        .filter(~F.col("status").isin(EXCLUDED_STATUSES))
        .agg(F.lit("silver.orders (billable)").alias("source"),
             F.count("*").alias("order_count"),
             F.sum("amount").alias("revenue")))

d_tot = (spark.table(GOLD_DAILY)
        .agg(F.lit("gold.daily_revenue").alias("source"),
             F.sum("order_count").alias("order_count"),
             F.sum("revenue").alias("revenue")))

g_tot = (spark.table(GOLD_CUSTOMER)
        .agg(F.lit("gold.customer_revenue").alias("source"),
             F.sum("order_count").alias("order_count"),
             F.sum("revenue").alias("revenue")))

recon = s_tot.unionByName(d_tot).unionByName(g_tot)
display(recon)

vals = {r["source"]: (r["order_count"], r["revenue"]) for r in recon.collect()}
distinct = set(vals.values())
assert len(distinct) == 1, f"RECONCILIATION FAILED: {vals}"

orders_n, revenue_n = distinct.pop()
print(f"RECONCILED — {orders_n} orders, {revenue_n} revenue, three independent paths")